In [1]:
# Import required libraries
import datetime
import json
import pandas as pd
import requests

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# 1. EXTRACT: Fetch raw data from GitHub API
print("Starting Extraction...")

# Calculate the exact date 30 days ago
thirty_days_ago = (
    datetime.datetime.now() - datetime.timedelta(days=30)
).strftime("%Y-%m-%d")
print(f"Targeting repositories created after: {thirty_days_ago}")

# GitHub API URL for searching repositories
url = "https://api.github.com/search/repositories"

# Query parameters: repos created in last 30 days, sorted by stars, top 50 results
params = {
    "q": f"created:>{thirty_days_ago}",
    "sort": "stars",
    "order": "desc",
    "per_page": 50,
}

# Make the network call to GitHub
response = requests.get(url, params=params)

# Check if the connection was successful (Status Code 200 means OK)
if response.status_code == 200:
    raw_data = response.json()["items"]
    print(f"Extraction Successful! Retrieved {len(raw_data)} repositories.")
else:
    print(f"Failed to fetch data. Status Code: {response.status_code}")
    raw_data = []

Starting Extraction...
Targeting repositories created after: 2026-05-12
Extraction Successful! Retrieved 50 repositories.


In [3]:
# 2. TRANSFORM: Clean and filter the messy data
print("Starting Transformation...")

cleaned_repos = []

for repo in raw_data:
    # Handle missing language values gracefully
    language = repo.get("language")
    if not language:
        language = "Unknown"

    # Extract only the specific data points we care about
    repo_dictionary = {
        "Repository Name": repo.get("name"),
        "Owner": repo.get("owner", {}).get("login"),
        "URL": repo.get("html_url"),
        "Stars": repo.get("stargazers_count"),
        "Forks": repo.get("forks_count"),
        "Primary Language": language,
    }
    # Add this clean dictionary to our list
    cleaned_repos.append(repo_dictionary)

# Convert the list into a structured Dataframe (a table format)
df = pd.DataFrame(cleaned_repos)

print("Transformation Successful!")

# Display the first 5 rows of our clean data right inside Colab
df.head()

Starting Transformation...
Transformation Successful!


,Repository Name,Owner,URL,Stars,Forks,Primary Language
0,odysseus,pewdiepie-archdaemon,https://github.com/pewdiepie-archdaemon/odysseus,68289,8561,Python
1,sdk-js,unicity-astrid,https://github.com/unicity-astrid/sdk-js,8256,36,TypeScript
2,open-code-review,alibaba,https://github.com/alibaba/open-code-review,6221,341,Go
3,defending-code-reference-harness,anthropics,https://github.com/anthropics/defending-code-r...,5730,412,Python
4,zerolang,vercel-labs,https://github.com/vercel-labs/zerolang,4995,330,C++


In [4]:
# 3. LOAD: Save the cleaned data to a CSV file
output_filename = "trending_github_repositories.csv"

# Save the dataframe to the file system (index=False prevents an extra row-number column)
df.to_csv(output_filename, index=False)

print(f"Load Successful! File saved as: {output_filename}")

Load Successful! File saved as: trending_github_repositories.csv
